In [ ]:
import torch

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Takes about 3-4 minutes
!pip install -q transformers==4.44.0
!pip install -q datasets==2.20.0
!pip install -q peft==0.12.0
!pip install -q trl==0.9.6
!pip install -q bitsandbytes==0.43.3
!pip install -q accelerate==0.33.0
!pip install -q huggingface_hub

print("✅ All libraries installed!")

In [ ]:
from huggingface_hub import login

# Paste your HF token here
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxx"
HF_USERNAME = "your-username"   # your HF username
MODEL_REPO = f"{HF_USERNAME}/marathi-mitra-phi3"

login(token=HF_TOKEN)
print(f"✅ Logged in!")
print(f"✅ Model will be saved to: {MODEL_REPO}")

In [ ]:
import json
import urllib.request

# Load vocabulary_dataset.json directly from your GitHub repo
GITHUB_USERNAME = "your-github-username"
url = f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/marathi-mitra/main/data/vocabulary_dataset.json"

with urllib.request.urlopen(url) as response:
    data = json.loads(response.read().decode("utf-8"))

print(f"✅ Loaded {len(data)} examples from GitHub")
print(f"✅ Keys: {list(data[0].keys())}")
print(f"\nSample text field preview:")
print(data[0]['text'][:300])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

# 4-bit quantization — makes model fit in T4 VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Phi-3 Mini... (takes 3-5 mins)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✅ Model loaded!")
print(f"✅ Size in memory: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
# CRITICAL — run this BEFORE training
# This proves what the model could NOT do before fine-tuning
# Save these outputs — they go in your README

def generate(prompt_word, model, tokenizer):
    prompt = f"""### Instruction:
You are Marathi Mitra, a friendly Marathi teacher for kids. \
When given an English word, teach it in Marathi with the word \
in Devanagari script, pronunciation, a simple story sentence, \
and a fun fact. Always be encouraging and kid-friendly.

### Input:
Teach me the Marathi word for: {prompt_word}

### Response:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()


# Test 5 words — save these outputs!
test_words = ["butterfly", "mother", "rain", "elephant", "school"]
baseline_results = {}

print("=" * 60)
print("BASELINE RESULTS — BEFORE FINE-TUNING")
print("=" * 60)

for word in test_words:
    result = generate(word, model, tokenizer)
    baseline_results[word] = result
    print(f"\n📝 Word: {word}")
    print(f"Output: {result}")
    print("-" * 40)

# Save baseline to compare later
import json
with open("baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, ensure_ascii=False, indent=2)

print("\n✅ Baseline results saved!")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                    # rank
    lora_alpha=32,           # scaling
    target_modules=[
        "q_proj", "k_proj",
        "v_proj", "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Expected output:
# trainable params: ~2M || all params: ~3.8B || trainable%: ~0.05%
# Only training 0.05% of the model — that's QLoRA!

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# Convert to HuggingFace Dataset
hf_dataset = Dataset.from_list(data)
print(f"✅ Dataset ready: {len(hf_dataset)} examples")

# Training configuration
training_args = SFTConfig(
    output_dir="./marathi-mitra-v1",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    max_seq_length=512,
    dataset_text_field="text",
    warmup_ratio=0.1,
    report_to="none",         # disable wandb for now
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_dataset,
    tokenizer=tokenizer,
)

print("🚀 Starting training...")
print("Expected time: ~15-20 mins on T4 GPU")
print("Watch the loss — it should decrease each epoch\n")

trainer.train()

print("\n✅ Training complete!")

In [ ]:
print("=" * 60)
print("RESULTS AFTER FINE-TUNING")
print("=" * 60)

finetuned_results = {}

for word in test_words:
    result = generate(word, model, tokenizer)
    finetuned_results[word] = result
    print(f"\n📝 Word: {word}")
    print(f"Output: {result}")
    print("-" * 40)

In [ ]:
print("=" * 60)
print("BEFORE vs AFTER COMPARISON")
print("=" * 60)

for word in test_words:
    print(f"\n🔤 Word: {word.upper()}")
    print(f"\n❌ BEFORE:\n{baseline_results[word]}")
    print(f"\n✅ AFTER:\n{finetuned_results[word]}")
    print("=" * 60)

In [ ]:
print(f"Saving model to {MODEL_REPO}...")

# Save adapter weights + tokenizer
model.push_to_hub(MODEL_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(MODEL_REPO, token=HF_TOKEN)

print(f"✅ Model saved!")
print(f"✅ View at: https://huggingface.co/{MODEL_REPO}")